# Knowledge-base cleanup — Qistas watermark & stray page numbers

Found while reviewing eval set v2 in the browser: the ground-truth source text for Labor Law Article 65 contained the line **`حقوق قسطاس للتقنيات القانونية`** — a PDF watermark belonging to Qistas, the Jordanian legal-technology company whose published PDF was the source document. It is not part of the law.

## Why this matters beyond cosmetics

In Article 65 the watermark is interleaved **mid-sentence**, because the PDF drew it over the text and extraction read it in reading order:

```
... بناء على تقرير من الطبيب المعتمد      <- sentence cut off here
حقوق قسطاس للتقنيات القانونية           <- watermark injected
من قبل المؤسسات التي يقل عدد عمالها ...   <- sentence resumes
```

So the sentence "a report from the physician approved **by institutions employing fewer than twenty workers**" is split by an unrelated company name. That damages three things at once: the embedding for the article (noise inside the vector), the BM25 term statistics, and the context string handed to the generator — which could in principle quote the watermark back to a user as if it were legal text.

## Scope

Deliberately narrow. Only `worklaw_ar_articles.xlsx` is affected:

| artefact | affected |
|---|---|
| Qistas watermark lines | Articles **37** and **65** |
| bare page-number lines (e.g. `٣٩ / ١٢`) | Articles **37** and **65** |
| `chunks_articles.parquet` units | `labor96_ar_37`, `labor96_ar_65` (2 of 531) |

The Social Security Law and Professional Work Law files are clean, and so are all three English files. An earlier scan appeared to show 39 affected articles, but that was a false positive from matching the bare word `حقوق` ("rights"), which occurs naturally throughout legal text — worth recording, since a crude substring scan on legal Arabic will always over-report.

## What this notebook does, and what it deliberately does not

It rewrites the two affected articles in the source `.xlsx` (after writing a timestamped backup) and regenerates the two affected rows of `chunks_articles.parquet`. It does **not** touch Pinecone: re-embedding and re-upserting costs money and is left as an explicit, separately-approved step. Until that step runs, the live index still contains the watermarked vectors.

In [1]:
import re
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
KB = PROJECT / "0. cleaned_data" / "Knowledgebase"
TARGET = KB / "worklaw_ar_articles.xlsx"
CHUNKS = PROJECT / "0. cleaned_data" / "chunks_articles.parquet"

# a line is junk if it is *only* the watermark, or *only* a page number -- never strip
# a line that also carries legal text, or real content would be lost with it
WATERMARK = re.compile(r"^\s*(حقوق\s+)?قسطاس\s+للتقنيات\s+القانونية\s*$")
PAGENUM = re.compile(r"^\s*[٠-٩0-9]{1,4}\s*/\s*[٠-٩0-9]{1,4}\s*$")

def clean(text: str) -> str:
    kept = [ln for ln in str(text).split("\n")
            if not WATERMARK.match(ln) and not PAGENUM.match(ln)]
    return "\n".join(kept)

df = pd.read_excel(TARGET)
df["Article #"] = df["Article #"].astype(str).str.strip()
affected = [r["Article #"] for _, r in df.iterrows() if clean(r["Content"]) != str(r["Content"])]
print("articles that will change:", affected)

articles that will change: ['33', '37', '65']


In [2]:
# show exactly which lines are removed, so the change is auditable rather than blind
for a in affected:
    before = str(df[df["Article #"] == a].iloc[0]["Content"])
    removed = [ln for ln in before.split("\n")
               if WATERMARK.match(ln) or PAGENUM.match(ln)]
    print(f"--- Article {a}: removing {len(removed)} line(s)")
    for ln in removed:
        print(f"    {ln!r}")
    # confirm nothing else changed: character count must drop by exactly the removed lines
    after = clean(before)
    expected_drop = sum(len(ln) + 1 for ln in removed)
    print(f"    chars {len(before)} -> {len(after)} (dropped {len(before) - len(after)}, expected ~{expected_drop})")

--- Article 33: removing 1 line(s)
    '٣٩ / ١١'
    chars 1028 -> 1020 (dropped 8, expected ~8)
--- Article 37: removing 2 line(s)
    'قسطاس للتقنيات القانونية'
    '٣٩ / ١٢'
    chars 391 -> 358 (dropped 33, expected ~33)
--- Article 65: removing 1 line(s)
    'حقوق قسطاس للتقنيات القانونية'
    chars 487 -> 457 (dropped 30, expected ~30)


In [3]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = TARGET.with_name(f"{TARGET.stem}.backup_{stamp}.xlsx")
shutil.copy2(TARGET, backup)
print("backup written:", backup.name)

df["Content"] = df["Content"].apply(clean)
df.to_excel(TARGET, index=False)
print(f"cleaned {len(affected)} articles in {TARGET.name}")

check = pd.read_excel(TARGET)
assert not check["Content"].astype(str).str.contains("قسطاس", regex=False, na=False).any(), "watermark survived"
assert len(check) == len(df), "row count changed"
print("verified: no watermark remains,", len(check), "articles intact")

backup written: worklaw_ar_articles.backup_20260801_122519.xlsx
cleaned 3 articles in worklaw_ar_articles.xlsx
verified: no watermark remains, 142 articles intact


In [4]:
# Patch the two affected rows of the production chunk table in place. Full re-chunking
# (scripts/14. rechunking_articles.ipynb) would rebuild all 531 units; only 2 changed, so a
# targeted patch keeps every other chunk byte-identical and the diff reviewable.
chunks = pd.read_parquet(CHUNKS)
before_n = len(chunks)

patched = []
for a in affected:
    cid = f"labor96_ar_{a}"
    new_text = str(check[check["Article #"].astype(str).str.strip() == a].iloc[0]["Content"])
    mask = chunks.chunk_id == cid
    if mask.any():
        old_len = len(chunks.loc[mask, "text"].iloc[0])
        chunks.loc[mask, "text"] = new_text
        patched.append((cid, old_len, len(new_text)))

assert len(chunks) == before_n, "chunk count changed"
assert not chunks.text.str.contains("قسطاس", regex=False, na=False).any(), "watermark still in chunks"
chunks.to_parquet(CHUNKS, index=False)

for cid, o, n in patched:
    print(f"{cid}: {o} -> {n} chars")
print(f"\n{len(chunks)} units written to {CHUNKS.name}, watermark gone")

labor96_ar_33: 1028 -> 1020 chars


labor96_ar_37: 391 -> 358 chars
labor96_ar_65: 487 -> 457 chars

531 units written to chunks_articles.parquet, watermark gone


## Still outstanding — needs explicit approval, costs money

The Pinecone index `jordan-labor-law-articles` still holds the **old, watermarked vectors** for `labor96_ar_37` and `labor96_ar_65`. Fixing that means re-embedding those two units with Cohere `embed-v4.0` and upserting them, which is two embedding calls plus two upserts — a fraction of a cent, but a paid API call all the same, so it is not run here.

Until it runs, the live system is in a mixed state: the local corpus is clean, the served vectors are not. BM25 runs off the local parquet and is therefore already fixed.

Worth noting for the extraction pipeline: the root cause is in Phase 1 (`scripts/01. files_text_extraction.ipynb`), which read the watermark as body text. A line-level filter of this kind belongs there so that any future re-extraction does not reintroduce it.